# Camillion — JAX/TPU Trainer (ONE bot, ALL symbols at once, pass FTMO consistently)

This trains **one bot that trades the WHOLE FTMO book at once** — it holds positions across every
symbol simultaneously, all sharing ONE equity/drawdown pot — on a TPU, playing **thousands of trading
lifetimes in parallel**, until it passes **40 FTMO challenges in a row on held-out data**. Every policy
+ its details + a live progress ledger are saved to your Google Drive.

**It is the SAME bot as the CPU trainer, written for the TPU.** A parity test proves the TPU env matches
the CPU env bar-for-bar (same 517 observation, contract v1.8.0, same reward, same FTMO rules) before training.

**The real run is Step 8b — the shared-pot PORTFOLIO bot on all your symbols.** Step 8 (single symbol)
is just an optional fast smoke/debug on the verified foundation. **Run order:** Runtime → Change runtime
type → **TPU**, then run top to bottom (you can skip Step 8 and go straight to 8b).

## 0. Confirm the TPU is connected (run this FIRST)


In [ ]:
# If this prints CpuDevice, STOP: Runtime -> Change runtime type -> TPU, then Reconnect.
import os
try:
    import jax
    print('jax', jax.__version__, '| devices:', jax.devices())
except Exception as e:
    print('jax not installed yet (cell 2 installs it). Detail:', e)


## 1. Mount Google Drive (where your data + saved policies live)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/Camillion'
DATA_DIR   = '/content/drive/MyDrive/Camillion_data'   # your 1-minute CSVs (one per symbol)
SAVE_DIR   = DRIVE_ROOT + '/jax_models'                # policies + jax_progress.jsonl land here
CACHE_DIR  = '/content/data_cache'                     # prepared market features (local, fast)
FEATURE_CACHE = DRIVE_ROOT + '/feature_cache'          # reusable per-symbol feature cache (Drive)
os.makedirs(SAVE_DIR, exist_ok=True); os.makedirs(CACHE_DIR, exist_ok=True)
print('SAVE_DIR  =', SAVE_DIR)
print('DATA_DIR  =', DATA_DIR, '(exists:', os.path.isdir(DATA_DIR), ')')


## 2. Install JAX for TPU + Flax/Optax (+ ONNX export tools)


In [ ]:
# Colab TPU uses a custom JAX build. Pin a recent jax[tpu]; flax/optax/onnx for the rest.
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','jax[tpu]',
                '-f','https://storage.googleapis.com/jax-releases/libtpu_releases.html'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','flax','optax','onnxscript','onnxruntime'], check=True)
import jax
print('jax', jax.__version__, '| devices:', jax.devices(), '| n_devices:', jax.local_device_count())
assert any('Tpu' in type(d).__name__ for d in jax.devices()), 'No TPU! set Runtime->TPU and re-run cell 0-2.'


## 3. Get the code (the `jax_tpu/` folder lives in the repo)


In [ ]:
import os, sys, subprocess
REPO='/content/Camillion_Claude_RL_model'
BRANCH='feat/jarvis-bridge'   # the branch that contains jax_tpu/
if not os.path.exists(REPO):
    subprocess.run(['git','clone','https://github.com/monty313/Camillion_Claude_RL_model.git',
                    '--branch',BRANCH,REPO], check=True)
sys.path.insert(0, REPO); os.chdir(REPO)
print('repo ready at', REPO)


## 4. PARITY GATE — prove the TPU env matches the CPU env (must pass before training)

If this fails, do NOT train: the TPU bot would not be the same bot. (Runs on CPU in seconds.)

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable,'jax_tpu/tests/run_parity.py'], capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-1500:])
assert 'passed' in r.stdout and 'FAIL' not in r.stdout, 'PARITY FAILED — stop and fix before training.'
print('>>> PARITY OK — the TPU env == the CPU env. Safe to train.')


## 5. Prepare market features from your Drive CSVs (the same path as run_training.py)

Pick ONE symbol to train (the single-symbol env is the verified foundation). This reuses the repo's
existing cache builder, so the data is identical to the CPU pipeline.


In [ ]:
SYMBOL = 'EURUSD'    # change to any symbol whose CSV is in DATA_DIR (e.g. XAUUSD, US30, GBPUSD)
BALANCE = 100_000.0  # account size (position sizes scale to it)
from run_training import prepare_caches
found = prepare_caches(DATA_DIR, [SYMBOL], CACHE_DIR)
assert found, f'No CSV for {SYMBOL} in {DATA_DIR} (put e.g. {SYMBOL}_1m.csv there).'
print('prepared:', found)


## 6. Build the shared static feature tensor (once) and move it onto the TPU

The CPU env precomputes indicators/alphas/streaks once; the TPU env then just **indexes** that shared
tensor (the 'build once + share' design). 459 of the 499 obs floats are byte-identical to the CPU env
(v1.6.0: the 20-float raw-OHLC block is one of the static blocks placed here).

In [ ]:
import numpy as np
from config.ftmo_config import load_ftmo_config
from config import asset_specs as A
from src.data.cache_builder import load_cache, load_aux
from src.env.trading_env import TradingEnv
from src.strategies.registry import AlphaRegistry
from src.strategies.alpha_pack import register_all
from src.data import feature_cache as FC
from jax_tpu import jax_static_features as JSF

import config.variables as V; V.STARTING_BALANCE = float(BALANCE)
cfg = load_ftmo_config()
ind, close, time_ns = load_cache(CACHE_DIR, SYMBOL)
aux = load_aux(CACHE_DIR, SYMBOL)   # v1.6.0: raw OHLC obs block (20) + ADX-DI side-channel (12)
psize = A.calibrated_position_size(SYMBOL, account=BALANCE) if SYMBOL in A.SPECS else BALANCE
reg = AlphaRegistry(); register_all(reg)
# load saved features from Drive if present (skips the slow precompute), else build + save.
# aux is part of the cache fingerprint -> a no-OHLC cache is never loaded as an OHLC one.
cached = FC.load(FEATURE_CACHE, SYMBOL, ind, close, time_ns, reg, aux=aux)
env = TradingEnv(ind, close, time_ns, reg, cfg=cfg, symbol=SYMBOL, position_size=psize,
                 warmup=200, precomputed=cached, progress=(cached is None), aux=aux)
if cached is None:
    try: FC.save(FEATURE_CACHE, SYMBOL, ind, close, time_ns, reg, env); print('saved feature cache ->', FEATURE_CACHE)
    except Exception as e: print('(feature cache save skipped:', e, ')')
sd = JSF.build_static_data(env)
print(f'{SYMBOL}: {sd.T:,} bars | static tensor {sd.static_obs.shape} '
      f'({sd.static_obs.nbytes/1e9:.2f} GB) | position_size={psize:,.2f}')


## 7. TPU utilization probe + autoscale to ~70-80%

We size the env army (`N_ENVS_PER_CORE`) so the TPU's matrix engine is busy evaluating the tiny
3x256 net for a huge batch. This cell runs a couple of timed iterations at increasing sizes and
suggests a value that lands utilization in the 70-80% band (more envs = better gradients = more work,
used well). Bump `N_ENVS_PER_CORE` until step time stops scaling sub-linearly or HBM ~80%.


In [ ]:
import time, jax, numpy as np
from jax_tpu import jax_config as JC, jax_env as JE, jax_trainer as TR
static = JE.make_device_static(sd)
params = JE.params_from_static(sd)
n_dev = jax.local_device_count()

def hbm_used_gb():
    try:
        s = jax.devices()[0].memory_stats(); return s.get('bytes_in_use',0)/1e9, s.get('bytes_limit',0)/1e9
    except Exception: return None, None

def time_iters(n_per_core, n_steps, iters=3):
    JC.N_ENVS_PER_CORE=n_per_core; JC.N_STEPS=n_steps
    ti, opt = TR.make_train_iter(static, params, sd.warmup, int(sd.T*0.8),
                                 int(min(JC.MAX_BARS, max(2,(int(sd.T*0.8)-sd.warmup)//2))))
    import jax.numpy as jnp; from jax_tpu import jax_ppo as PPO
    key=jax.random.PRNGKey(0); _,p=PPO.init_params(key); nm=PPO.norm_init()
    rep=lambda t: jax.tree_util.tree_map(lambda x: jnp.broadcast_to(x,(n_dev,)+jnp.asarray(x).shape), t)
    pp,po,pn=rep(p),rep(opt.init(p)),rep(nm)
    ks=jax.random.split(key,n_dev); st,ob=jax.vmap(lambda k: TR._fresh_states(k,n_per_core,static,params,sd.warmup,int(sd.T*0.8),int(min(JC.MAX_BARS,max(2,(int(sd.T*0.8)-sd.warmup)//2)))))(ks)
    dk=jax.random.split(jax.random.PRNGKey(1),n_dev); pe=jnp.zeros((n_dev,))+0.01
    out=ti(pp,po,pn,st,ob,dk,pe); jax.block_until_ready(out[0]); t0=time.time()
    for _ in range(iters):
        out=ti(*out[:6],pe); jax.block_until_ready(out[0])
    dt=(time.time()-t0)/iters; bs=n_dev*n_per_core*n_steps
    u,lim=hbm_used_gb(); print(f'  envs/core={n_per_core:5d} steps={n_steps} batch={bs:>10,} | {dt*1e3:7.1f} ms/iter | {bs/dt/1e6:7.2f}M states/s'+(f' | HBM {u:.1f}/{lim:.1f} GB ({100*u/lim:.0f}%)' if u else ''))
    return dt, bs, (u,lim)

print(f'devices={n_dev}. Probing scale (raise until throughput plateaus or HBM ~80%)...')
for npc in [256, 512, 1024, 2048]:
    try: time_iters(npc, JC.N_STEPS)
    except Exception as e: print(f'  envs/core={npc}: stopped ({type(e).__name__}: {str(e)[:80]}) -> back off'); break
print('\nPick the largest envs/core that still fit comfortably (HBM ~70-80%) for the training cell.')


## 8. (OPTIONAL) Single-symbol smoke train — the verified foundation

This trains on ONE symbol. It's a fast sanity check that the on-device PPO loop, the live dashboard,
and the Drive checkpoints all work. **The actual goal is Step 8b (ALL symbols at once)** — you can skip
straight there. `resume=True`, so a disconnect just means re-run the cell.


In [ ]:
from jax_tpu import jax_trainer as TR, jax_config as JC, jax_progress as PROG
JC.N_ENVS_PER_CORE = 2048   # <- set from the probe (use the largest that fit ~70-80% HBM)
JC.N_STEPS = 128
# LIVE PROGRESS: a dashboard that redraws every eval so you WATCH P(pass) @ 2.5%/4% and the
# '40 challenge passes in a row' streak climb as training runs (operator request).
dash = PROG.LiveDashboard(target=40)
details = TR.train(
    sd, save_dir=SAVE_DIR, resume=True,
    target_streak=40,         # operator: 40 challenge passes in a row (held-out)
    eval_every=50,            # evaluate + checkpoint every 50 updates
    on_eval=dash,             # <-- live FTMO-consistency dashboard (P(pass) + streak-to-40)
    log_every=10,             # light heartbeat line between evals
    env_param_kwargs=dict(    # the locked FTMO numbers (training domain-randomizes risk around these)
        daily_target_frac=0.025, trailing_dd_frac=0.04, daily_dd_frac=0.05,
        total_dd_frac=0.10, profit_target_frac=0.10, two_phase_enabled=1.0,
        phase2_continue=1.0, phase2_trailing_frac=0.01),
)
print('FINAL:', details)


## 8b. ★ THE GOAL — train ONE bot on ALL symbols at once (shared-pot portfolio)

This trains the real product: **one bot trading the entire book simultaneously** — long EURUSD, short
GBPUSD, long XAUUSD… all at once, every position drawing on ONE shared equity/drawdown pot. It trains
**UNBOUNDED — it does NOT stop until the policy passes 40 FTMO challenges IN A ROW on held-out data**
(re-run on disconnect; `resume=True` continues). The live dashboard redraws every eval so you watch:

- **P(pass) @ 2.5%/4%** and the **40-passes-in-a-row** bar climbing,
- a **DIAGNOSIS** (HIDING / OVER-TRADING / LEARNING — and which knob to turn),
- **VS ALPHAS** — whether the bot is BEATING or merely following the alpha consensus,
- **activity** + **breach rate** so you see *how* it's getting there.


In [ ]:
from jax_tpu import jax_static_features as JSF, jax_trainer as TR, jax_config as JC, jax_progress as PROG
# Build ONE TradingEnv per symbol, time-align, stack to the portfolio tensor.
PORT_SYMBOLS = ['EURUSD','GBPUSD','XAUUSD','US30']   # whichever CSVs are in DATA_DIR
from run_training import prepare_caches
from src.data.cache_builder import load_cache, load_aux
from src.env.portfolio_env import align_symbol_data, build_portfolio_subs
from src.strategies.registry import AlphaRegistry
from src.strategies.alpha_pack import register_all
import config.variables as V; V.STARTING_BALANCE = float(BALANCE)
found = prepare_caches(DATA_DIR, PORT_SYMBOLS, CACHE_DIR)
assert found, f'no CSVs for {PORT_SYMBOLS} in {DATA_DIR}'
def _reg():
    r = AlphaRegistry(); register_all(r); return r
# 4-tuple per symbol so the v1.6.0 aux (OHLC obs block + ADX-DI side-channel) flows into the subs.
sd_map = align_symbol_data({s: (*load_cache(CACHE_DIR, s), load_aux(CACHE_DIR, s)) for s in found})
# risk_pct = the per-symbol BASE size cap (4 symbols x 0.6% ~ 2.4%/day worst case, inside the 4% wall).
subs = build_portfolio_subs(sd_map, _reg, warmup=200, feature_cache_dir=FEATURE_CACHE, risk_pct=0.6)
psd = JSF.build_portfolio_static(subs)
print(f'portfolio: {psd.N} symbols x {psd.T:,} bars | static {psd.static_obs.shape} ({psd.static_obs.nbytes/1e9:.1f} GB)')

JC.N_ENVS_PER_CORE = 1024   # raise via the probe (cell 7) toward ~70-80% HBM
JC.N_STEPS = 128
dash = PROG.LiveDashboard(target=40)
# TRAINS UNBOUNDED: stops ONLY when the bot strings 40 WINNING DAYS IN A ROW on held-out data
# (a winning day ends >= +2.5%; a BREACH or a losing day RESETS the streak -> start over). resume=True
# survives disconnects. Watch the dashboard: winning-days bar, DIAGNOSIS, VS ALPHAS, ACTIONS, SYMBOLS.
details = TR.train_portfolio(
    psd, save_dir=SAVE_DIR + '/portfolio', resume=True,
    target_streak=40,          # 40 winning days in a row (held-out) = the stop
    eval_every=50, on_eval=dash, log_every=10,
    env_param_kwargs=dict(     # FTMO rules + the tunable reward knobs (tune from the DIAGNOSIS line):
        daily_target_frac=0.025, trailing_dd_frac=0.04, daily_dd_frac=0.05, total_dd_frac=0.10,
        profit_target_frac=0.10, two_phase_enabled=1.0, phase2_continue=1.0, phase2_trailing_frac=0.01,
        continue_after_pass=1.0,            # keep trading past +10% (we want long won-day streaks)
        breach_penalty=20.0,                # 2x a won day, felt immediately (operator 2026-06-29)
        dd_proximity_coef=2.0,              # scaled-up: nearing the 4% wall costs real points (plan away from it)
        target_seek_weight=3.0, idle_day_penalty=0.02,    # SEEK the +2.5% target (scaled to stay visible); don't HIDE
        alpha_on=1.0, alpha_agree=0.01, alpha_against=0.01, alpha_beat=0.05,   # 10x: develop edge (BEAT), don't just follow
        # WON day = +10 + an ESCALATING streak bonus (+1 per ADDITIONAL consecutive won day, capped at +10);
        # failed day = -5 + streak reset. Replaces the old every-4th jackpot. pass_bonus = the +10% terminal (eval).
        day_pass_reward=10.0, day_fail_penalty=5.0, streak_bonus=1.0, streak_bonus_cap=10.0, pass_bonus=1.0,
        # v1.7.0 TRADE-RISK: BB(10,1) HARD STOP + RISK-BASED sizing (~0.1%/trade off the stop, capped at
        # the calibrated base) + small PnL-capped band-stack & re-entry CLOSE bonuses (operator 2026-06-29).
        bb_stop_enabled=1.0, risk_based=1.0, risk_frac=0.001,
        band_stack_bonus=0.005, reentry_bonus=0.003,
        # CONVICTION nudge: >=2 of the 3 strong-setup alphas (slots 18/19/20: CCI|>160| 5m+30m,
        # BB200&BB20 double-breakout any-TF, fwd-SMA(4) 5m+30m) confirmed the entry AND it won. PnL-capped;
        # kept MODEST on purpose (a big pay-to-trade bonus fights 40-in-a-row). Crank if you want more pull.
        conviction_bonus=0.1,
        # 5m CCI open-gate (operator 2026-06-29): DON'T trade the chop -> block a new open when BOTH the
        # 5m CCI(30) AND CCI(100) sit inside +/-50 (no momentum). Lets the bot take a protected zero on a
        # dead 5m instead of forcing a trade.
        open_gate=1.0),
)
print('PORTFOLIO FINAL:', details)


## 9. Progress report (read the ledger Drive saved as we trained)


In [ ]:
import json, os
path = os.path.join(SAVE_DIR, 'jax_progress.jsonl')
rows = [json.loads(l) for l in open(path)] if os.path.exists(path) else []
print(f'{len(rows)} eval rows logged at {path}')
for r in rows[-12:]:
    print(f"  update {r['update']:>6} | reward {r['mean_reward']:+.5f} | pass_rate {r['eval_pass_rate']:.1%} "
          f"| streak {r['consecutive_passes']:>2}/40 (best {r['best_streak_global']}) | {r.get('iters_per_s','?')} it/s")
try:
    import matplotlib.pyplot as plt
    u=[r['update'] for r in rows]
    fig,ax=plt.subplots(1,2,figsize=(12,3.5))
    ax[0].plot(u,[r['eval_pass_rate'] for r in rows]); ax[0].set_title('held-out pass rate'); ax[0].set_xlabel('update')
    ax[1].plot(u,[r['consecutive_passes'] for r in rows]); ax[1].axhline(40,ls='--',c='r'); ax[1].set_title('consecutive passes (target 40)'); ax[1].set_xlabel('update')
    plt.tight_layout(); plt.show()
except Exception as e: print('(plot skipped:', e, ')')


## 10. Export the best policy to ONNX (for MT5 / live)

Converts the JAX policy to a PyTorch 3x256 net (normalization baked in) and exports ONNX — the same
format the CPU bot deploys with. Verified bit-close to the JAX policy before writing.


In [ ]:
from jax_tpu import export_to_pytorch as EXP
import os
# export the rolling best (or use 'passed_40_in_a_row' once the gate is hit)
tag = 'passed_40_in_a_row' if os.path.isdir(os.path.join(SAVE_DIR,'passed_40_in_a_row')) else 'best_policy'
out = os.path.join(SAVE_DIR, f'camillion_jax_{SYMBOL}.onnx')
EXP.convert(SAVE_DIR, tag, out)
print('exported policy:', out)
